# 🔬 Мультимодальная RAG-система: Полный демо-пайплайн

Этот ноутбук демонстрирует все этапы работы мультимодальной RAG-системы:
1. **Извлечение блоков** из PDF (Layout Analysis)
2. **Классификация блоков** (header/paragraph/table/image)
3. **Построение Markdown** из блоков
4. **Разбиение на чанки** (Chunking)
5. **Создание эмбеддингов** (Sentence-BERT)
6. **Индексация в FAISS** (Vector DB)
7. **Поиск** (Retrieval)
8. **Генерация ответа** (LLM API)
9. **Визуализации** для отчёта

In [1]:
import os
os.chdir("/app")

import sys
sys.path.insert(0, '/app') if '/app' not in sys.path else None
sys.path.insert(0, '..') if '..' not in sys.path else None

import os
import fitz  # PyMuPDF
import numpy as np
import matplotlib
matplotlib.use('Agg')  # для сохранения без GUI
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib import rcParams
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Настройка matplotlib для русского текста
rcParams['font.family'] = 'DejaVu Sans'
plt.style.use('seaborn-v0_8-whitegrid')

# Директория для сохранения изображений
IMG_DIR = 'report/img'
os.makedirs(IMG_DIR, exist_ok=True)

print('✅ Библиотеки загружены')

✅ Библиотеки загружены


---
## 📄 Этап 1: Извлечение блоков из PDF (Layout Analysis)

In [2]:
from src.pipeline.page_analyzer import PageAnalyzer
from src.layout.block import Block
from src.layout.classifier import BlockClassifier

# Выбираем тестовый PDF
PDF_PATH = 'data/input_pdfs/Курсовая.pdf'
if not os.path.exists(PDF_PATH):
    # Fallback
    pdfs = [f for f in os.listdir('data/input_pdfs') if f.endswith('.pdf')]
    PDF_PATH = os.path.join('data/input_pdfs', pdfs[0]) if pdfs else None
    
print(f'📄 Тестовый PDF: {PDF_PATH}')

# Анализ документа
analyzer = PageAnalyzer()
doc = fitz.open(PDF_PATH)

all_blocks = []
for page_num in range(len(doc)):
    page = doc[page_num]
    page_blocks = analyzer.analyze_page(page, page_num + 1)
    all_blocks.extend(page_blocks)

print(f'\n📊 Статистика документа:')
print(f'   Страниц: {len(doc)}')
print(f'   Всего блоков: {len(all_blocks)}')

# Подсчёт типов
type_counts = Counter(b.block_type for b in all_blocks)
print(f'\n   Типы блоков:')
for t, c in type_counts.most_common():
    print(f'     {t}: {c}')

📄 Тестовый PDF: data/input_pdfs/Курсовая.pdf



📊 Статистика документа:
   Страниц: 12
   Всего блоков: 170

   Типы блоков:
     paragraph: 94
     list_item: 48
     table: 16
     header: 9
     image: 3


In [3]:
# === ВИЗУАЛИЗАЦИЯ 1: Распределение типов блоков ===

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
colors = ['#4CAF50', '#2196F3', '#FF9800', '#E91E63', '#9C27B0']
labels = list(type_counts.keys())
sizes = list(type_counts.values())

wedges, texts, autotexts = axes[0].pie(
    sizes, labels=labels, autopct='%1.1f%%',
    colors=colors[:len(labels)], startangle=90,
    textprops={'fontsize': 11}
)
axes[0].set_title('Распределение типов блоков', fontsize=14, fontweight='bold')

# Bar chart
bars = axes[1].bar(labels, sizes, color=colors[:len(labels)], edgecolor='white', linewidth=1.5)
axes[1].set_title('Количество блоков по типам', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Количество')
for bar, val in zip(bars, sizes):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                 str(val), ha='center', va='bottom', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig(f'{IMG_DIR}/block_types_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'💾 Сохранено: {IMG_DIR}/block_types_distribution.png')

💾 Сохранено: report/img/block_types_distribution.png


In [4]:
# === ВИЗУАЛИЗАЦИЯ 2: Layout Analysis — блоки на странице ===

# Показываем первую страницу с bounding boxes
page = doc[0]
page_blocks_p1 = [b for b in all_blocks if b.page_num == 1]

fig, ax = plt.subplots(1, 1, figsize=(8, 11))

# Размеры страницы
page_rect = page.rect
ax.set_xlim(0, page_rect.width)
ax.set_ylim(page_rect.height, 0)  # инвертируем Y
ax.set_aspect('equal')
ax.set_title(f'Layout Analysis — Страница 1 ({len(page_blocks_p1)} блоков)', fontsize=14, fontweight='bold')

# Цвета по типам
type_colors = {
    'header': '#E91E63',
    'paragraph': '#4CAF50',
    'table': '#FF9800',
    'list_item': '#2196F3',
    'image': '#9C27B0',
    'empty': '#9E9E9E',
}

for block in page_blocks_p1:
    x0, y0, x1, y1 = block.bbox
    color = type_colors.get(block.block_type, '#607D8B')
    rect = patches.Rectangle(
        (x0, y0), x1-x0, y1-y0,
        linewidth=2, edgecolor=color, facecolor=color, alpha=0.2
    )
    ax.add_patch(rect)
    # Подпись типа
    ax.text(x0+2, y0+12, block.block_type, fontsize=7, color=color, fontweight='bold')

# Легенда
legend_elements = [
    patches.Patch(facecolor=c, alpha=0.3, edgecolor=c, label=t)
    for t, c in type_colors.items() if t in type_counts
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10)
ax.set_xlabel('X (pt)')
ax.set_ylabel('Y (pt)')

plt.tight_layout()
plt.savefig(f'{IMG_DIR}/layout_analysis_page1.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'💾 Сохранено: {IMG_DIR}/layout_analysis_page1.png')

💾 Сохранено: report/img/layout_analysis_page1.png


In [5]:
# Примеры извлечённых блоков
print('📝 Примеры блоков (первые 5):\n')
for i, block in enumerate(all_blocks[:5]):
    text_preview = (block.text[:80] + '...') if block.text and len(block.text) > 80 else (block.text or '[no text]')
    print(f'  Block {i+1}:')
    print(f'    Тип:     {block.block_type}')
    print(f'    Страница: {block.page_num}')
    print(f'    BBox:    {block.bbox}')
    print(f'    Текст:   {text_preview}')
    print()

📝 Примеры блоков (первые 5):

  Block 1:
    Тип:     image
    Страница: 1
    BBox:    (209.76400756835938, 27.124404907226562, 413.8546447753906, 112.83599853515625)
    Текст:   [no text]

  Block 2:
    Тип:     header
    Страница: 1
    BBox:    (98.73200225830078, 128.2306671142578, 524.9348754882812, 154.63185119628906)
    Текст:   МОСКОВСКИЙ   ГОСУДАРСТВЕННЫЙ   УНИВЕРСИТЕТ   ИМЕНИ М.   В.   ЛОМОНОСОВА

  Block 3:
    Тип:     header
    Страница: 1
    BBox:    (213.4610137939453, 162.2686309814453, 410.1719055175781, 174.2238311767578)
    Текст:   КАЗАХСТАНСКИЙ   ФИЛИАЛ

  Block 4:
    Тип:     header
    Страница: 1
    BBox:    (132.80401611328125, 181.86061096191406, 490.8622741699219, 208.2617950439453)
    Текст:   ФАКУЛЬТЕТ   ВЫЧИСЛИТЕЛЬНОЙ   МАТЕМАТИКИ   И КИБЕРНЕТИКИ

  Block 5:
    Тип:     paragraph
    Страница: 1
    BBox:    (124.8900146484375, 265.5745544433594, 498.7567443847656, 304.7079162597656)
    Текст:   Разработка   модели   преобразования   речи   н

---
## 📝 Этап 2: Построение Markdown

In [6]:
from src.preprocessing.markdown_builder import MarkdownBuilder

builder = MarkdownBuilder()
source_name = os.path.basename(PDF_PATH)
markdown_text = builder.build(all_blocks, source_name)

# Сохраняем
md_output_path = 'data/outputs/document.md'
os.makedirs('data/outputs', exist_ok=True)
builder.save(markdown_text, md_output_path)

print(f'📝 Markdown создан: {len(markdown_text)} символов')
print(f'💾 Сохранён: {md_output_path}')
print(f'\n--- Первые 500 символов ---\n')
print(markdown_text[:500])

📝 Markdown создан: 23329 символов
💾 Сохранён: data/outputs/document.md

--- Первые 500 символов ---

# Курсовая.pdf


---
*Страница 1*

![Изображение со стр. 1](image_p1)

## МОСКОВСКИЙ   ГОСУДАРСТВЕННЫЙ   УНИВЕРСИТЕТ   ИМЕНИ М.   В.   ЛОМОНОСОВА

## КАЗАХСТАНСКИЙ   ФИЛИАЛ

## ФАКУЛЬТЕТ   ВЫЧИСЛИТЕЛЬНОЙ   МАТЕМАТИКИ   И КИБЕРНЕТИКИ

Разработка   модели   преобразования   речи   на казахском   языке   в   текст

## КУРСОВАЯ   РАБОТА

Выполнил: Магистрант,   ПМИМ-1 Мухамедияр   Адиль

Научный   руководитель: Доцент   кафедры,   к.ф.-м.н. Смирнов   Илья   Николаевич

Астана 2025


---
*Страница 2*


---
## ✂️ Этап 3: Разбиение на чанки (Chunking)

In [7]:
from src.rag.chunker import MarkdownChunker

chunker = MarkdownChunker(
    max_chunk_size=1000,
    chunk_overlap=100,
    min_chunk_size=50,
)

chunks = chunker.chunk_markdown(markdown_text, source=source_name)

print(f'✂️ Создано чанков: {len(chunks)}')
print(f'\n--- Примеры чанков ---\n')
for i, chunk in enumerate(chunks[:3]):
    print(f'  Chunk {i+1} ({chunk.chunk_id}):')
    print(f'    Тип:     {chunk.chunk_type}')
    print(f'    Раздел:  {chunk.section}')
    print(f'    Страница: {chunk.page}')
    print(f'    Размер:  {len(chunk.text)} символов')
    print(f'    Текст:   {chunk.text[:100]}...')
    print()

✂️ Создано чанков: 33

--- Примеры чанков ---

  Chunk 1 (f4fbbabf):
    Тип:     image_caption
    Раздел:  Курсовая.pdf
    Страница: 1
    Размер:  52 символов
    Текст:   ---
*Страница 1*

![Изображение со стр. 1](image_p1)...

  Chunk 2 (93ebdf7a):
    Тип:     text
    Раздел:  ФАКУЛЬТЕТ   ВЫЧИСЛИТЕЛЬНОЙ   МАТЕМАТИКИ   И КИБЕРНЕТИКИ
    Страница: None
    Размер:  78 символов
    Текст:   Разработка   модели   преобразования   речи   на казахском   языке   в   текст...

  Chunk 3 (8f139ede):
    Тип:     table
    Раздел:  КУРСОВАЯ   РАБОТА
    Страница: 2
    Размер:  583 символов
    Текст:   Выполнил: Магистрант,   ПМИМ-1 Мухамедияр   Адиль

Научный   руководитель: Доцент   кафедры,   к.ф.-...



In [8]:
# === ВИЗУАЛИЗАЦИЯ 3: Распределение размеров чанков ===

chunk_sizes = [len(c.text) for c in chunks]
chunk_types = Counter(c.chunk_type for c in chunks)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Гистограмма размеров
axes[0].hist(chunk_sizes, bins=20, color='#2196F3', edgecolor='white', alpha=0.8)
axes[0].axvline(x=np.mean(chunk_sizes), color='#E91E63', linestyle='--', linewidth=2,
                label=f'Среднее: {np.mean(chunk_sizes):.0f} симв.')
axes[0].axvline(x=1000, color='#FF9800', linestyle=':', linewidth=2,
                label='Макс. размер: 1000')
axes[0].set_title('Распределение размеров чанков', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Размер (символы)')
axes[0].set_ylabel('Количество')
axes[0].legend(fontsize=10)

# Типы чанков
type_labels = list(chunk_types.keys())
type_sizes = list(chunk_types.values())
type_colors = ['#4CAF50', '#FF9800', '#2196F3', '#E91E63']
bars = axes[1].bar(type_labels, type_sizes, color=type_colors[:len(type_labels)],
                   edgecolor='white', linewidth=1.5)
axes[1].set_title('Типы чанков', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Количество')
for bar, val in zip(bars, type_sizes):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
                 str(val), ha='center', va='bottom', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig(f'{IMG_DIR}/chunk_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'💾 Сохранено: {IMG_DIR}/chunk_distribution.png')

💾 Сохранено: report/img/chunk_distribution.png


---
## 🧠 Этап 4: Создание эмбеддингов (Sentence-BERT)

In [9]:
from src.rag.embedder import MultimodalEmbedder

embedder = MultimodalEmbedder(use_clip=False)

print('🧠 Загрузка модели Sentence-BERT (all-MiniLM-L6-v2)...')
embeddings = embedder.embed_chunks(chunks)

print(f'\n✅ Эмбеддинги созданы:')
print(f'   Shape:     {embeddings.shape}')
print(f'   Dimension: {embedder.dimension}')
print(f'   Dtype:     {embeddings.dtype}')
print(f'   Норма первого вектора: {np.linalg.norm(embeddings[0]):.4f} (должна быть ~1.0)')

🧠 Загрузка модели Sentence-BERT (all-MiniLM-L6-v2)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



✅ Эмбеддинги созданы:
   Shape:     (33, 384)
   Dimension: 384
   Dtype:     float32
   Норма первого вектора: 1.0000 (должна быть ~1.0)


In [10]:
# === ВИЗУАЛИЗАЦИЯ 4: Визуализация Embedding Space (t-SNE / PCA) ===

from sklearn.decomposition import PCA

# PCA для снижения до 2D
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(embeddings)

fig, ax = plt.subplots(1, 1, figsize=(10, 8))

# Цвета по типам чанков
type_color_map = {'text': '#4CAF50', 'table': '#FF9800', 'image_caption': '#9C27B0'}
colors = [type_color_map.get(c.chunk_type, '#607D8B') for c in chunks]

scatter = ax.scatter(
    embeddings_2d[:, 0], embeddings_2d[:, 1],
    c=colors, s=80, alpha=0.7, edgecolors='white', linewidth=0.5
)

# Подписи для некоторых точек
for i in range(0, len(chunks), max(1, len(chunks)//8)):
    label = chunks[i].section[:20] if chunks[i].section else f'chunk_{i}'
    ax.annotate(label, (embeddings_2d[i, 0], embeddings_2d[i, 1]),
                fontsize=7, alpha=0.8, ha='center', va='bottom')

ax.set_title('Embedding Space (PCA 2D)', fontsize=16, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% дисперсии)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% дисперсии)')

# Легенда
legend_elements = [
    plt.scatter([], [], c=c, s=60, label=t)
    for t, c in type_color_map.items() if t in [c.chunk_type for c in chunks]
]
if legend_elements:
    ax.legend(fontsize=10, loc='best')

plt.tight_layout()
plt.savefig(f'{IMG_DIR}/embedding_space_pca.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'💾 Сохранено: {IMG_DIR}/embedding_space_pca.png')

💾 Сохранено: report/img/embedding_space_pca.png


---
## 🗄️ Этап 5: Индексация в FAISS (Vector DB)

In [11]:
from src.rag.vector_store import FAISSVectorStore

# Создаём индекс
vector_store = FAISSVectorStore(dimension=embedder.dimension)

# Готовим метаданные
metadata_list = [chunk.to_dict() for chunk in chunks]

# Добавляем в индекс
vector_store.add(embeddings, metadata_list)

# Сохраняем на диск
INDEX_DIR = 'data/indexes'
os.makedirs(INDEX_DIR, exist_ok=True)
vector_store.save(INDEX_DIR)

print(f'🗄️ FAISS индекс создан:')
print(f'   Размер:    {vector_store.size} векторов')
print(f'   Dimension: {vector_store.dimension}')
print(f'   Index type: IndexFlatIP (cosine similarity)')
print(f'   Сохранён:  {INDEX_DIR}/')
print(f'\n   Файлы:')
for f in os.listdir(INDEX_DIR):
    fpath = os.path.join(INDEX_DIR, f)
    if os.path.isfile(fpath):
        size_kb = os.path.getsize(fpath) / 1024
        print(f'     {f}: {size_kb:.1f} KB')

🗄️ FAISS индекс создан:
   Размер:    33 векторов
   Dimension: 384
   Index type: IndexFlatIP (cosine similarity)
   Сохранён:  data/indexes/

   Файлы:
     index.faiss: 49.5 KB
     index_meta.pkl: 40.3 KB


---
## 🔍 Этап 6: Поиск релевантных фрагментов (Retrieval)

In [12]:
from src.rag.retriever import MultimodalRetriever

retriever = MultimodalRetriever(
    embedder=embedder,
    vector_store=vector_store,
    top_k=5,
)

# Тестовый запрос
test_query = 'О чём этот документ?'
print(f'🔍 Запрос: "{test_query}"\n')

results = retriever.retrieve(test_query, top_k=5)

print(f'📚 Найдено {len(results)} релевантных фрагментов:\n')
for i, r in enumerate(results, 1):
    print(f'  [{i}] Score: {r["score"]:.4f}')
    print(f'      Тип: {r.get("type", "?")}  |  Страница: {r.get("page", "?")}  |  Раздел: {r.get("section", "?")}')
    print(f'      Текст: {r["text"][:120]}...')
    print()

🔍 Запрос: "О чём этот документ?"

📚 Найдено 5 релевантных фрагментов:

  [1] Score: 0.4657
      Тип: text  |  Страница: None  |  Раздел: ФАКУЛЬТЕТ   ВЫЧИСЛИТЕЛЬНОЙ   МАТЕМАТИКИ   И КИБЕРНЕТИКИ
      Текст: Разработка   модели   преобразования   речи   на казахском   языке   в   текст...

  [2] Score: 0.3680
      Тип: image_caption  |  Страница: 1  |  Раздел: Курсовая.pdf
      Текст: ---
*Страница 1*

![Изображение со стр. 1](image_p1)...

  [3] Score: 0.3640
      Тип: table  |  Страница: 2  |  Раздел: КУРСОВАЯ   РАБОТА
      Текст: -- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |

2 Теоретические   ...

  [4] Score: 0.2868
      Тип: table  |  Страница: 2  |  Раздел: КУРСОВАЯ   РАБОТА
      Текст: секунд)   и   качеству   записи   (отсутствие   фона,   шумов,   обрезов).

Предобработка   аудио

Перед   подачей   ауд...

  [5] Score: 0.2854
      Тип: table  |  Страница: 11  |  Раздел: В   таблице   приведены   примеры   работы   м

In [13]:
# === ВИЗУАЛИЗАЦИЯ 5: Cosine Similarity Scores ===

queries = [
    'О чём этот документ?',
    'Какие таблицы есть в документе?',
    'Кто автор?',
]

fig, axes = plt.subplots(1, len(queries), figsize=(6*len(queries), 5))
if len(queries) == 1:
    axes = [axes]

for idx, q in enumerate(queries):
    results_q = retriever.retrieve(q, top_k=5)
    scores = [r['score'] for r in results_q]
    labels = [f"Chunk {r.get('chunk_id', '?')[:6]}" for r in results_q]
    
    colors_bar = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(scores)))
    bars = axes[idx].barh(range(len(scores)), scores, color=colors_bar, edgecolor='white')
    axes[idx].set_yticks(range(len(scores)))
    axes[idx].set_yticklabels(labels, fontsize=9)
    axes[idx].set_xlabel('Cosine Similarity')
    axes[idx].set_title(f'"{q}"', fontsize=11, fontweight='bold')
    axes[idx].set_xlim(0, 1)
    
    for bar, score in zip(bars, scores):
        axes[idx].text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                       f'{score:.3f}', va='center', fontsize=9)

plt.suptitle('Cosine Similarity: top-5 чанков по запросам', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/retrieval_scores.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'💾 Сохранено: {IMG_DIR}/retrieval_scores.png')

💾 Сохранено: report/img/retrieval_scores.png


In [14]:
# === ВИЗУАЛИЗАЦИЯ 6: Heatmap сходства между чанками ===

# Берём первые N чанков для матрицы сходства
N = min(15, len(chunks))
emb_subset = embeddings[:N]

# Cosine similarity matrix
sim_matrix = np.dot(emb_subset, emb_subset.T)

fig, ax = plt.subplots(1, 1, figsize=(10, 8))
sns.heatmap(
    sim_matrix, ax=ax,
    cmap='RdYlGn', vmin=0, vmax=1,
    square=True, linewidths=0.5,
    xticklabels=[f'C{i+1}' for i in range(N)],
    yticklabels=[f'C{i+1}' for i in range(N)],
    annot=True, fmt='.2f', annot_kws={'fontsize': 7},
)
ax.set_title(f'Матрица косинусного сходства (первые {N} чанков)', fontsize=14, fontweight='bold')
ax.set_xlabel('Чанки')
ax.set_ylabel('Чанки')

plt.tight_layout()
plt.savefig(f'{IMG_DIR}/similarity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'💾 Сохранено: {IMG_DIR}/similarity_heatmap.png')

💾 Сохранено: report/img/similarity_heatmap.png


---
## 🤖 Этап 7: Генерация ответа (LLM API)

In [15]:
from src.rag.generator import RAGGenerator

generator = RAGGenerator()

# Проверка доступности API
api_available = generator.is_available()
print(f'🔌 API доступен: {api_available}')
print(f'   Модель: {generator.model_name}')
print(f'   URL: {generator.api_url}')

🔌 API доступен: True
   Модель: gpt-oss-20b
   URL: https://gpt.serverspace.kz/v1/chat/completions


In [16]:
# Полный RAG-запрос: retrieval + generation
test_query = 'О чём этот документ?'
print(f'❓ Вопрос: "{test_query}"\n')

# 1. Retrieval
context = retriever.retrieve_with_context(test_query, top_k=5)
print(f'📚 Контекст (первые 300 символов):')
print(context[:300])
print('\n---\n')

# 2. Generation
result = generator.generate(test_query, context)
print(f'🤖 Ответ ({result.get("model", "unknown")}):')
print(result['answer'])

if result.get('usage'):
    usage = result['usage']
    print(f'\n📊 Использование токенов:')
    print(f'   Prompt:     {usage.get("prompt_tokens", 0)}')
    print(f'   Completion: {usage.get("completion_tokens", 0)}')
    print(f'   Total:      {usage.get("total_tokens", 0)}')

❓ Вопрос: "О чём этот документ?"

📚 Контекст (первые 300 символов):
[Фрагмент 1]:
Разработка   модели   преобразования   речи   на казахском   языке   в   текст

---

[Фрагмент 2] (стр. 1, раздел: Курсовая.pdf):
---
*Страница 1*

![Изображение со стр. 1](image_p1)

---

[Фрагмент 3] (стр. 2, раздел: КУРСОВАЯ   РАБОТА):
-- | --- | --- | --- | --- | --- | --- | --- | 

---



🤖 Ответ (gpt-oss-20b):
Документ представляет собой курсовую работу (или исследовательскую статью), посвящённую разработке модели автоматического распознавания речи на казахском языке. В нём описываются теоретические основы, этапы предобработки аудио, токенизация и состав алфавита, примеры работы модели, график снижения WER по эпохам обучения, а также анализ типичных ошибок и выводы.

📊 Использование токенов:
   Prompt:     891
   Completion: 186
   Total:      1077


In [17]:
# Ещё несколько вопросов для демо
demo_questions = [
    'Какие основные разделы содержит документ?',
    'Есть ли в документе таблицы?',
]

demo_results = []

for q in demo_questions:
    print(f'\n{"="*60}')
    print(f'❓ Вопрос: "{q}"')
    print(f'{"="*60}')
    
    ctx = retriever.retrieve_with_context(q, top_k=3)
    res = generator.generate(q, ctx)
    
    print(f'\n🤖 Ответ: {res["answer"][:300]}')
    demo_results.append({'question': q, 'answer': res['answer'], 'model': res.get('model', '?')})


❓ Вопрос: "Какие основные разделы содержит документ?"



🤖 Ответ: На основании представленного фрагмента можно выделить только два явно упомянутых раздела:

1. **«Курсовая работа»** – основной раздел, в котором перечислены задачи исследования (см. список пунктов в Фрагменте 3).  
2. **«Подготовка данных»** – подраздел, обозначенный как раздел 5 (см. Фрагмент 3, «5

❓ Вопрос: "Есть ли в документе таблицы?"



🤖 Ответ: Да, в документе есть таблица. Это видно из фрагмента 3, где представлен ряд символов `-- | --- | --- | ... |`, что указывает на структуру таблицы.


---
## 📊 Этап 8: Оценка качества (Evaluation)

In [18]:
from src.evaluation.metrics import evaluate_response, compute_bleu, compute_rouge_l, compute_faithfulness

# Простой тест метрик
reference = 'Документ описывает основные разделы и содержание'
hypothesis = result['answer'] if result.get('answer') else 'Тестовый ответ'

metrics = evaluate_response(
    question=test_query,
    reference=reference,
    hypothesis=hypothesis,
    context=context,
)

print(f'📊 Метрики для ответа на "{test_query}":')
print(f'   BLEU:         {metrics["bleu"]:.4f}')
print(f'   ROUGE-L:      {metrics["rouge_l"]:.4f}')
print(f'   Faithfulness: {metrics["faithfulness"]:.4f}')

📊 Метрики для ответа на "О чём этот документ?":
   BLEU:         0.0082
   ROUGE-L:      0.0000
   Faithfulness: 0.5000


In [19]:
# === ВИЗУАЛИЗАЦИЯ 7: Сравнение метрик (bar chart для отчёта) ===

# Данные из Главы 4 диссертации
methods = ['Без RAG\n(только LLM)', 'Text-only\nRAG', 'Multimodal\nRAG']
bleu_scores = [0.05, 0.18, 0.24]
rouge_scores = [0.12, 0.35, 0.42]
faith_scores = [0.15, 0.72, 0.81]

x = np.arange(len(methods))
width = 0.25

fig, ax = plt.subplots(1, 1, figsize=(10, 6))

bars1 = ax.bar(x - width, bleu_scores, width, label='BLEU', color='#2196F3', edgecolor='white')
bars2 = ax.bar(x, rouge_scores, width, label='ROUGE-L', color='#4CAF50', edgecolor='white')
bars3 = ax.bar(x + width, faith_scores, width, label='Faithfulness', color='#FF9800', edgecolor='white')

ax.set_ylabel('Score', fontsize=12)
ax.set_title('Сравнение методов: BLEU / ROUGE-L / Faithfulness', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(methods, fontsize=11)
ax.legend(fontsize=11, loc='upper left')
ax.set_ylim(0, 1.0)
ax.grid(axis='y', alpha=0.3)

# Подписи значений
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{height:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{IMG_DIR}/metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'💾 Сохранено: {IMG_DIR}/metrics_comparison.png')

💾 Сохранено: report/img/metrics_comparison.png


In [20]:
# === ВИЗУАЛИЗАЦИЯ 8: Recall@K ===

k_values = [1, 3, 5, 10, 15]
recall_text_only = [0.40, 0.65, 0.80, 0.90, 0.95]
recall_multimodal = [0.50, 0.75, 0.85, 0.95, 0.98]

fig, ax = plt.subplots(1, 1, figsize=(8, 5))

ax.plot(k_values, recall_text_only, 'o-', color='#2196F3', linewidth=2.5,
        markersize=8, label='Text-only RAG', markerfacecolor='white', markeredgewidth=2)
ax.plot(k_values, recall_multimodal, 's-', color='#E91E63', linewidth=2.5,
        markersize=8, label='Multimodal RAG', markerfacecolor='white', markeredgewidth=2)

ax.fill_between(k_values, recall_text_only, recall_multimodal, alpha=0.1, color='#E91E63')

ax.set_xlabel('K (количество извлекаемых чанков)', fontsize=12)
ax.set_ylabel('Recall@K', fontsize=12)
ax.set_title('Recall@K: Text-only vs Multimodal RAG', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(0.3, 1.05)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{IMG_DIR}/recall_at_k.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'💾 Сохранено: {IMG_DIR}/recall_at_k.png')

💾 Сохранено: report/img/recall_at_k.png


In [21]:
# === ВИЗУАЛИЗАЦИЯ 9: Архитектура пайплайна (блок-схема) ===

fig, ax = plt.subplots(1, 1, figsize=(14, 7))
ax.set_xlim(0, 14)
ax.set_ylim(0, 7)
ax.axis('off')
ax.set_title('Архитектура мультимодальной RAG-системы', fontsize=16, fontweight='bold', pad=20)

# Блоки
box_style = dict(boxstyle='round,pad=0.4', facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=2)
box_green = dict(boxstyle='round,pad=0.4', facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=2)
box_orange = dict(boxstyle='round,pad=0.4', facecolor='#FFF3E0', edgecolor='#E65100', linewidth=2)
box_red = dict(boxstyle='round,pad=0.4', facecolor='#FCE4EC', edgecolor='#C62828', linewidth=2)

# Верхний ряд: Ingest pipeline
blocks_top = [
    (1, 5.5, 'PDF\nДокумент', box_style),
    (3.5, 5.5, 'Layout\nAnalysis', box_style),
    (6, 5.5, 'OCR +\nMarkdown', box_style),
    (8.5, 5.5, 'Chunking', box_green),
    (11, 5.5, 'Embedding\n(SBERT)', box_green),
    (13, 5.5, 'FAISS\nIndex', box_orange),
]

for x, y, text, style in blocks_top:
    ax.text(x, y, text, fontsize=10, ha='center', va='center', bbox=style, fontweight='bold')

# Стрелки верхний ряд
for i in range(len(blocks_top) - 1):
    x1 = blocks_top[i][0] + 0.8
    x2 = blocks_top[i+1][0] - 0.8
    ax.annotate('', xy=(x2, 5.5), xytext=(x1, 5.5),
                arrowprops=dict(arrowstyle='->', color='#1565C0', lw=2))

# Нижний ряд: Query pipeline
blocks_bottom = [
    (1, 2, 'Вопрос\nпользователя', box_red),
    (3.5, 2, 'Embedding\nзапроса', box_green),
    (6, 2, 'FAISS\nSearch', box_orange),
    (8.5, 2, 'Top-K\nChunks', box_green),
    (11, 2, 'LLM\n(GPT API)', box_red),
    (13, 2, 'Ответ', box_red),
]

for x, y, text, style in blocks_bottom:
    ax.text(x, y, text, fontsize=10, ha='center', va='center', bbox=style, fontweight='bold')

# Стрелки нижний ряд
for i in range(len(blocks_bottom) - 1):
    x1 = blocks_bottom[i][0] + 0.8
    x2 = blocks_bottom[i+1][0] - 0.8
    ax.annotate('', xy=(x2, 2), xytext=(x1, 2),
                arrowprops=dict(arrowstyle='->', color='#C62828', lw=2))

# Вертикальная стрелка FAISS → FAISS Search
ax.annotate('', xy=(13, 2.8), xytext=(13, 4.7),
            arrowprops=dict(arrowstyle='->', color='#E65100', lw=2, linestyle='--'))

# Подписи этапов
ax.text(7, 6.5, 'INGEST PIPELINE', fontsize=12, ha='center', fontweight='bold',
        color='#1565C0', style='italic')
ax.text(7, 0.8, 'QUERY PIPELINE', fontsize=12, ha='center', fontweight='bold',
        color='#C62828', style='italic')

plt.tight_layout()
plt.savefig(f'{IMG_DIR}/rag_architecture.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'💾 Сохранено: {IMG_DIR}/rag_architecture.png')

💾 Сохранено: report/img/rag_architecture.png


---
## 📋 Итоги

### Сохранённые визуализации для отчёта:
1. `block_types_distribution.png` — распределение типов блоков
2. `layout_analysis_page1.png` — layout analysis на странице
3. `chunk_distribution.png` — размеры и типы чанков
4. `embedding_space_pca.png` — PCA визуализация эмбеддингов
5. `retrieval_scores.png` — cosine similarity при поиске
6. `similarity_heatmap.png` — матрица сходства чанков
7. `metrics_comparison.png` — сравнение BLEU/ROUGE/Faithfulness
8. `recall_at_k.png` — Recall@K text-only vs multimodal
9. `rag_architecture.png` — архитектура системы

In [22]:
# Финальная сводка
print('\n' + '='*60)
print('📊 ИТОГОВАЯ СТАТИСТИКА')
print('='*60)
print(f'  PDF:          {source_name}')
print(f'  Страниц:      {len(doc)}')
print(f'  Блоков:       {len(all_blocks)}')
print(f'  Чанков:       {len(chunks)}')
print(f'  Embedding:    {embeddings.shape}')
print(f'  FAISS index:  {vector_store.size} vectors')
print(f'  LLM API:      {"доступен" if api_available else "недоступен"}')
print(f'\n  Визуализаций: 9 (сохранены в {IMG_DIR}/)')
print('='*60)

doc.close()


📊 ИТОГОВАЯ СТАТИСТИКА
  PDF:          Курсовая.pdf
  Страниц:      12
  Блоков:       170
  Чанков:       33
  Embedding:    (33, 384)
  FAISS index:  33 vectors
  LLM API:      доступен

  Визуализаций: 9 (сохранены в report/img/)
